In [ ]:
# ============================================================
# Notebook 7: "Constraining the Shadow"
# Regression from the Inside: Seeing the Geometry of Linear Models
# ============================================================

# --- Environment setup (run this cell first) ---
import sys

# Install regression_geometry package if not available
try:
    import regression_geometry
except ImportError:
    print("Installing regression_geometry package...")
    !pip install -q git+https://github.com/YOUR_USERNAME/regression-geometry-course.git
    print("Done! If you see import errors below, restart the runtime (Runtime → Restart) and run this cell again.")

# --- Standard imports ---
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg

import statsmodels.api as sm

from regression_geometry.core import ColumnSpace, Projection, Ellipsoid
from regression_geometry.data import load_meridian
from regression_geometry.colors import *

# --- Rendering backend toggle ---
INTERACTIVE = True
try:
    from regression_geometry import interactive as viz_mod
    if not viz_mod.AVAILABLE:
        INTERACTIVE = False
except ImportError:
    INTERACTIVE = False

if INTERACTIVE:
    from regression_geometry import interactive as viz
else:
    from regression_geometry import plots as viz

from regression_geometry.scoreboard import GeometricScoreboard

# --- Reproducibility ---
np.random.seed(42)

> *"A shorter leash does not make the dog weaker — it makes the dog predictable."*

## The Problem: When "Optimal" Isn't Good Enough

Notebook 6 showed you what happens when the column space degenerates: the eigenvalue ellipsoid stretches into a needle, the projection skates along the thin direction, and small perturbations cause wild coefficient swings. The condition number exploded. The Scoreboard went red.

The problem wasn't the projection itself — it was optimal. The problem was that "optimal" meant "closest point on a razor's edge." What if you deliberately stepped AWAY from the closest point, accepting a small amount of bias in exchange for stability?

That's regularization. And it has its own geometry.

In [ ]:
# Reproduce the collinear scenario from Notebook 6 (same seed)
rng = np.random.default_rng(42)
n = 100
x1 = rng.normal(0, 1, n)
x2 = 0.95 * x1 + np.sqrt(1 - 0.95**2) * rng.normal(0, 1, n)  # r(x1,x2) ~ 0.95
beta_true = np.array([3.0, 2.0, 1.5])  # intercept, x1, x2
y_collinear = beta_true[0] + beta_true[1] * x1 + beta_true[2] * x2 + rng.normal(0, 1, n)

X_collinear = sm.add_constant(np.column_stack([x1, x2]))
model_collinear = sm.OLS(y_collinear, X_collinear).fit()
X_no_intercept = np.column_stack([x1, x2])
ell = Ellipsoid(X_no_intercept.T @ X_no_intercept)

print(f"Correlation r(x1, x2): {np.corrcoef(x1, x2)[0,1]:.3f}")
print(f"OLS coefficients:      {model_collinear.params.round(3)}")
print(f"Standard errors:       {model_collinear.bse.round(3)}")
print(f"\nEigenvalues:      {ell.eigenvalues.round(2)}")
print(f"Condition number: {ell.condition_number:.1f}")

fig = viz.plot_eigenvalue_ellipsoid(ell, title="Eigenvalue Ellipsoid — Collinear Data")

## Ridge as Constrained Projection

OLS says: find the point in the column space closest to y. No restrictions.

Ridge says: find the point closest to y, BUT the coefficients must stay inside a sphere of radius t. If the OLS solution is inside the sphere, Ridge and OLS agree. If OLS is outside the sphere, Ridge pulls the solution back to the sphere's surface.

The sphere is the constraint. The solution is where the RSS contour ellipse first touches the sphere.

In [ ]:
# Visualize Ridge vs. LASSO constraint boundaries in 2D coefficient space
beta_ols_2d = model_collinear.params[1:]  # exclude intercept
fig = viz.plot_ridge_lasso_constraint(beta_ols_2d,
    title="Ridge (L2 circle) vs. LASSO (L1 diamond)")

The OLS solution sits outside the purple circle. Ridge finds the point ON the circle that minimizes RSS — where the smallest ellipse just touches the circle. The Ridge solution is biased (it's not the OLS point) but it's constrained (it can't swing wildly).

---
### 🛑 PREDICT FIRST

As you tighten the constraint (shrink the circle), the Ridge coefficients change. But they don't all shrink at the same rate.

**Before running the next cell, write your prediction below:**

Do all coefficients shrink equally, or do some shrink faster? What determines which coefficients shrink most — their size, their stability, or something else?

---

In [ ]:
# YOUR PREDICTION:
#
#

In [ ]:
# Ridge estimator via eigendecomposition
# beta_ridge = Q diag(lambda_k / (lambda_k + lam)) Q' beta_OLS
lam = 10.0
shrinkage = ell.shrinkage_factors(lam)
beta_ridge = ell.ridge_coefficients(beta_ols_2d, lam)

print(f"Eigenvalues of X'X:        {ell.eigenvalues.round(2)}")
print(f"Shrinkage factors (lam={lam}): {shrinkage.round(4)}")
print()
print(f"  Large eigenvalue -> factor {shrinkage[0]:.4f} -> barely shrunk (stable direction)")
print(f"  Small eigenvalue -> factor {shrinkage[1]:.4f} -> aggressively shrunk (unstable direction)")
print()
print(f"OLS coefficients:   {beta_ols_2d.round(4)}")
print(f"Ridge coefficients: {beta_ridge.round(4)}")

Ridge is smart. It doesn't shrink everything equally. It identifies the UNSTABLE directions — where the eigenvalues are small, where the column space is thin — and shrinks those most aggressively. The stable directions barely change. Ridge is performing *selective geometric surgery* on exactly the dimensions that cause trouble.

In [ ]:
# Shrinkage path: coefficient values as a function of lambda
fig = viz.plot_shrinkage_path(ell, beta_ols_2d,
    title="Ridge Shrinkage Path — Collinear Data")

In [ ]:
# Interactive Ridge/LASSO explorer
if INTERACTIVE:
    display(viz.plot_ridge_lasso_interactive(X_no_intercept, y_collinear,
        title="Regularization Explorer"))
else:
    fig = viz.plot_ridge_lasso_constraint(beta_ols_2d,
        title="Ridge vs. LASSO at fixed constraint")

Follow any single coefficient line in the shrinkage path. It moves smoothly from its OLS value toward zero. No jumps. No sudden changes. Ridge is a continuous operation.

**Here the full geometry matters.** In Notebooks 1-5, you used the Relevant Triangle principle: every scalar quantity reduces to a 2D or 3D picture. That principle breaks down here. Ridge shrinkage depends on ALL eigenvalues simultaneously — not just a pairwise relationship. The shrinkage factor lambda_k/(lambda_k + lambda) involves the entire spectrum of X'X.

This is one of two places in the course (the other is the F-test in Notebook 8) where the low-dimensional intuition is not sufficient.

## LASSO — Sparsity as a Corner Solution

Ridge uses an L2 constraint — a sphere. The surface is smooth everywhere. The solution slides smoothly as lambda changes, and coefficients approach zero but never reach it exactly.

LASSO uses an L1 constraint — a diamond. The surface has CORNERS. And something remarkable happens at those corners.

In [ ]:
# Show Ridge (circle) vs. LASSO (diamond) side by side
fig = viz.plot_ridge_lasso_constraint(beta_ols_2d,
    title="Ridge (smooth circle) vs. LASSO (cornered diamond)")

The RSS ellipse touches the diamond at a corner. At a corner, one coordinate is exactly zero. That coefficient has been eliminated from the model. This is why LASSO produces sparse solutions — not because of any special algebraic trick, but because of the geometry of touching a diamond. Ellipses are more likely to touch diamonds at corners than at edges.

---
### 🛑 PREDICT FIRST

In 3D (three coefficients), the L1 constraint is an octahedron — a shape with corners (where two coefficients are zero, one survives), edges (where one coefficient is zero), and faces (where none are zero).

**Before running the next cell, write your prediction below:**

As you tighten the LASSO constraint, where will the solution land FIRST — on a face, an edge, or a corner? What does each location mean for the number of surviving predictors?

---

In [ ]:
# YOUR PREDICTION:
#
#

In [ ]:
# Ridge vs. LASSO on the collinear data
from sklearn.linear_model import Ridge, Lasso

lam_compare = 5.0
ridge_model = Ridge(alpha=lam_compare, fit_intercept=True).fit(
    X_no_intercept, y_collinear)
lasso_model = Lasso(alpha=lam_compare / (2 * n), fit_intercept=True,
    max_iter=10000).fit(X_no_intercept, y_collinear)

print(f"OLS:   x1 = {model_collinear.params[1]:.4f},  x2 = {model_collinear.params[2]:.4f}")
print(f"Ridge: x1 = {ridge_model.coef_[0]:.4f},  x2 = {ridge_model.coef_[1]:.4f}")
print(f"LASSO: x1 = {lasso_model.coef_[0]:.4f},  x2 = {lasso_model.coef_[1]:.4f}")

When predictors are correlated, Ridge keeps both and shrinks them. LASSO drops one and keeps the other. Ridge says "both matter a little." LASSO says "I'll pick one." Neither is universally better — it depends on whether you believe the true model is dense (many small effects) or sparse (few large effects).

In [ ]:
# Regularization on Meridian salary regression
df = load_meridian()
predictors = ["experience", "education", "performance"]
X_mer = df[predictors].values
y_mer = df["salary"].values

X_mer_sm = sm.add_constant(X_mer)
model_ols_mer = sm.OLS(y_mer, X_mer_sm).fit()

ridge_mer = Ridge(alpha=1.0, fit_intercept=True).fit(X_mer, y_mer)
lasso_mer = Lasso(alpha=1.0, fit_intercept=True, max_iter=10000).fit(X_mer, y_mer)

print(f"{'Method':<8} {'experience':>12} {'education':>12} {'performance':>12}")
print(f"{'OLS':<8} {model_ols_mer.params[1]:>12.2f} {model_ols_mer.params[2]:>12.2f} {model_ols_mer.params[3]:>12.2f}")
print(f"{'Ridge':<8} {ridge_mer.coef_[0]:>12.2f} {ridge_mer.coef_[1]:>12.2f} {ridge_mer.coef_[2]:>12.2f}")
print(f"{'LASSO':<8} {lasso_mer.coef_[0]:>12.2f} {lasso_mer.coef_[1]:>12.2f} {lasso_mer.coef_[2]:>12.2f}")

On the Meridian data, OLS already works fine — the column space is well-conditioned. Regularization doesn't change the story much. But if you added 20 more noisy predictors, LASSO would start dropping them. Regularization shines when you have many predictors relative to observations — a geometry problem where the column space is bloated and the projection is overwhelmed.

<details>
<summary><b>Going Deeper:</b> Ridge as a Bayesian posterior mode</summary>

If you place a Normal prior on beta with mean zero and variance sigma-squared/lambda, the posterior mode equals the Ridge estimate. The L2 constraint sphere IS the prior distribution.

Larger lambda = tighter prior = more shrinkage toward zero. The Ridge objective minimize ||y - X beta||^2 + lambda ||beta||^2 has the same solution as the Bayesian posterior mode under Gaussian likelihood and Gaussian prior. The regularization parameter lambda controls the prior precision — how strongly you believe the coefficients should be near zero before seeing the data.

</details>

---
### 🛑 DIAGNOSE FIRST

You've seen that Ridge shrinks coefficients smoothly and LASSO can zero them out. Look at the Meridian comparison table above.

**Before seeing the visualization, answer:**

Which Meridian predictor does LASSO shrink the most? What does that tell you about its relationship with salary, after controlling for the other variables?

---

In [ ]:
# YOUR ANSWER:
#
#

---
### 🔗 Back to Statsmodels

| Geometric concept | Code |
|---|---|
| Ridge coefficients (constrained projection) | `Ridge(alpha=lam).fit(X, y).coef_` |
| LASSO coefficients (corner solution) | `Lasso(alpha=lam).fit(X, y).coef_` |
| Regularization path | `sklearn.linear_model.lasso_path(X, y)` |
| Eigendecomposition of Ridge | `ell.ridge_coefficients(beta_ols, lam)` |

---

In [ ]:
# Verify: sklearn Ridge matches our eigendecomposition
lam_check = 10.0
ridge_check = Ridge(alpha=lam_check, fit_intercept=False).fit(
    X_no_intercept, y_collinear)

cs_collinear = ColumnSpace(X_no_intercept, add_intercept=False)
proj_collinear = cs_collinear.project(y_collinear)
beta_geo = ell.ridge_coefficients(proj_collinear.coefficients, lam_check)

print(f"sklearn Ridge:    {ridge_check.coef_.round(6)}")
print(f"Geometric Ridge:  {beta_geo.round(6)}")
print(f"Difference:       {np.max(np.abs(ridge_check.coef_ - beta_geo)):.2e}")

---
### ✍️ The Memo

You're writing a memo to Meridian's Head of Analytics recommending whether to use OLS, Ridge, or LASSO for a new prediction model with 50 candidate variables.

In 3 sentences, explain the trade-off. When would you use each method?

**Rules:** Do not use the words "eigenvalue," "constraint boundary," "L1 norm," "L2 norm," or "sparsity." Write in plain English that the Head of Analytics (who knows basic regression but not regularization) would understand.

---

In [ ]:
# YOUR MEMO:
#
#
#

In [ ]:
# Geometric Scoreboard — all 5 gauges active
cs_mer = ColumnSpace(X_mer_sm, add_intercept=False)
proj_mer = cs_mer.project(y_mer)

scoreboard = GeometricScoreboard(
    proj=proj_mer,
    cs=cs_mer,
    active_gauges=["theta", "r_squared", "leverage", "residual_norm", "kappa"],
    mode="widget" if INTERACTIVE else "static"
)
scoreboard.display()

All five gauges are active. Note that kappa (condition number) is healthy for Meridian — this is a well-conditioned problem where regularization doesn't change much. On the collinear data, kappa would be in the red zone, and that's exactly where Ridge earns its keep.

---
### Summary

**What you learned:**
- Ridge constrains the projection to a sphere, shrinking coefficients smoothly toward zero — with the most unstable directions shrunk most aggressively.
- LASSO constrains to a diamond, producing exact zeros at the corners — automatic variable selection.
- Neither is an orthogonal projection. Both trade a small amount of bias for a large reduction in variance.

**Key geometric insight:** ***Ridge shrinks the unstable eigenvalue directions. LASSO exploits the corners of the diamond. The geometry of the constraint determines the geometry of the solution.***

### Next

Notebook 8 returns to OLS and asks — how do you formally test whether the extra dimensions you added to the column space actually helped? The F-test compares two projections.

---